In [ ]:
import random 
import string
import requests
import csv
import numpy as np
from datetime import datetime, UTC
import time
import langid
import pandas as pd

### Choose subset to scrap

In [ ]:
X = 2

Load usernames to scrap

In [ ]:
with open(f'covid_users_{X}.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

df = pd.read_csv('data.csv', sep=',')
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(by=['user', "date"], ascending=True)

### Scrap content 
With this first script, we try to mitigate the number of contributions necessary to fill in the dataset. We gather contributions per username by request of 50. Then for each contribution, we require 1 additional request (post) or 2 additionnal requests (post + interlocutor's contribution, in case of response).

We gather all information except user's account creation date and interlocutors profile pictures. These can be requested separately later.

After every request, the results are saved in `data.csv` with link to the next page to request. 

We also implement a test to filter out english-speaking users: after the first request (50 contributions), we estimate the likelihood of being French. If it is too low, we discard the user.

In [ ]:
random_sleeps = np.random.uniform(low=0.35, high=0.65, size=100_000_000)

for i, username in enumerate(lines):
    username = username.strip('\n')
    
    after, to_cont = None, True
    j = (df['user']==username).sum() + 1

    # If username already in dataframe, load last 'after' property
    if j > 1:
        subset = df[df['user']==username]
        if 'End' in subset['after'].unique():
            to_cont = False
        after = subset.iloc[0]['after']

    
    headuser = "Mozilla/5.0 (compatible; scraper/1.0)"
    

    # Loop until all contributions have been scraped
    while to_cont:
        headuser += random.choice(string.ascii_uppercase + string.digits)
        headers = {
            "User-Agent": headuser
        }
        session = requests.Session()
        session.headers.update(headers)
        params = params = {
                "limit": 50
            }
        
        if after:
            params["after"] = after

        r = session.get(
                f"https://www.reddit.com/user/{username}/.json",
                params=params,
                timeout=3)
        data = r.json()

        if data["data"]["after"]:
            after = data["data"]["after"]
        else : 
            after = 'End'
            to_cont = False

        # For every contribution in the request, gather relevant informations
        if data['data']['children']:
            donnees_user = []
            batch_language = []
            for child in data['data']['children']:
                id = str(i+1).zfill(4) + str(j)
                prod = child['data']
                prod_created = datetime.fromtimestamp(prod['created_utc'], UTC)

                if child['kind']=='t1': # contribution = comment 

                    # gather information on the post
                    post_title = prod['link_title']
                    r = session.get(
                            f"https://www.reddit.com/api/info.json?id={prod['link_id']}",
                            timeout=3)
                    post = r.json()['data']['children'][0]['data']
                    post_body = str(post['selftext'].replace("\n", "\\n"))
                    parent_user = post['author']

                    body = str(prod['body'].replace("\n", "\\n"))

                    if prod['parent_id'].startswith('t1'): # comment to comment (= response)
                        type_prod = 'response'
                        r = session.get(
                            f"https://www.reddit.com/api/info.json?id={prod['parent_id']}",
                            timeout=3)
                        parent = r.json()['data']['children'][0]['data']
                        parent_user = parent['author']
                        parent_body = str(parent['body'].replace("\n", "\\n"))
                    elif prod['parent_id'].startswith('t3') : # Comment to post (=comment)
                        type_prod = 'comment'
                        parent_body = 'post body'
                    

                elif child['kind']=='t3': # contribution = post
                    type_prod = 'post'
                    body = str(prod['selftext'].replace("\n", "\\n"))
                    parent_user = np.nan
                    parent_body = np.nan
                    post_title = prod['title']
                    post_body = 'self body'
                
                # Estimate if the contribution is in french
                language, score = langid.classify(body)
                batch_language.append(language)

            
                interaction = [f'{X}X{id}', username, np.nan, type_prod, prod_created,
                    body, parent_user,  np.nan, parent_body, # np.nan corresponds to registration date and parent's pp
                    post_title, post_body, after, language] 
                donnees_user.append(interaction)

                j += 1
                if j%100 == 0:
                    time.sleep(random_sleeps[j])
                    print(body, j)

            # If few contributions are in english in first batch, then discard user
            if j<=51:
                if batch_language.count('fr') <=5:
                    with open("data.csv", "a", newline="", encoding="utf-8") as f:
                        writer = csv.writer(f)
                        writer.writerow([f'{X}X{id}', username] + [np.nan for k in range(9)] + ['End', 'en'])  
                    to_cont = False
                    continue

            # Save on data.csv
            with open("data.csv", "a", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerows(donnees_user)

    print(f'User {i+1} scraped with {j-1} contributions.')

In [ ]:
        # r = session.get(
        #         f"https://www.reddit.com/user/{username}/about.json", 
        #         params=params,
        #         timeout=3)
        # date_ins = date_regis = datetime.utcfromtimestamp(r.json()["data"]["created_utc"])
